# 🎙️ Deepfake Audio Detection System - Production Trainer

This notebook contains the **entire codebase** (except the API) for the Audio-Guard system. It is designed for high-performance training on Google Colab using **Wav2Vec2**.

### ✨ Key Features
- 📁 **Custom Datasets**: Support for your own local or Drive-based audio folders.
- 🧠 **Wav2Vec2 Architecture**: Fine-tuning transformer layers while freezing CNN features.
- 🧹 **Advanced Preprocessing**: Smart silence trimming, normalization, and chunking.
- 📊 **Rich Evaluation**: Confusion matrices, ROC curves, and detailed classification reports.
- 🚀 **Production Inference**: High-level wrapper for easy testing.

In [ ]:
# @title 🛠️ Step 1: Initialize Environment & Mount Drive
%matplotlib inline
!pip install -q datasets librosa soundfile pydub noisereduce transformers accelerate evaluate jiwer gtts pyttsx3 rich seaborn

import os
import sys
import time
import json
import random
import logging
import warnings
from pathlib import Path
from typing import List, Optional, Tuple, Dict, Union

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

from google.colab import drive
# drive.mount('/content/drive')  # Uncomment if your files are on Drive

console = Console()
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger(__name__)

print("✅ Environment ready.")

In [ ]:
# @title ⚙️ Step 2: Global Configuration
class Config:
    # ── Paths ────────────────────────────────
    BASE_DIR      = Path("./data")
    RAW_DIR       = BASE_DIR / "raw"
    PROCESSED_DIR = BASE_DIR / "processed"
    CHECKPOINTS   = Path("./checkpoints")
    RESULTS_DIR   = Path("./results")

    # ── Custom Datasets (UPDATE THESE!) ──────
    # You can point these to local folders or Drive paths
    CUSTOM_PATH_1 = "" # @param {type:"string"}
    CUSTOM_LABEL_1 = "real" # @param ["real", "fake"]
    
    CUSTOM_PATH_2 = "" # @param {type:"string"}
    CUSTOM_LABEL_2 = "real" # @param ["real", "fake"]

    # ── Audio ────────────────────────────────
    SAMPLE_RATE      = 16_000          # Wav2Vec2 expects 16kHz
    CHUNK_DURATION   = 4.0             # seconds per chunk
    MIN_DURATION_SEC = 1.0
    TARGET_DB        = -20.0
    TOP_DB           = 30

    # ── Model ────────────────────────────────
    MODEL_NAME        = "facebook/wav2vec2-base"
    NUM_LABELS        = 2
    HIDDEN_DROPOUT    = 0.1
    CLASSIFIER_DROPOUT = 0.3
    FREEZE_FEATURE_EXTRACTOR = True

    # ── Training ─────────────────────────────
    LEARNING_RATE  = 2e-5
    WEIGHT_DECAY   = 1e-4
    BATCH_SIZE     = 8
    EVAL_BATCH_SIZE = 16
    NUM_EPOCHS     = 6
    WARMUP_RATIO   = 0.1
    VAL_SPLIT      = 0.15
    SEED           = 42
    FP16           = True
    METRIC_FOR_BEST = "f1"

    LABEL_MAP = {0: "real", 1: "fake"}
    INV_LABEL_MAP = {"real": 0, "fake": 1}

cfg = Config()

# Ensure directories exist
for p in [cfg.RAW_DIR/"real", cfg.RAW_DIR/"fake", cfg.PROCESSED_DIR/"real", cfg.PROCESSED_DIR/"fake", cfg.CHECKPOINTS, cfg.RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("✅ Configuration applied.")

In [ ]:
# @title 🧹 Step 3: Audio Preprocessing Engine
import librosa
import numpy as np
import soundfile as sf

def load_audio(path: Path) -> np.ndarray:
    waveform, sr = librosa.load(str(path), sr=cfg.SAMPLE_RATE, mono=True)
    return waveform.astype(np.float32)

def transform_audio(waveform: np.ndarray) -> np.ndarray:
    # Trim silence
    waveform, _ = librosa.effects.trim(waveform, top_db=cfg.TOP_DB)
    # Peak-Normalize
    peak = np.abs(waveform).max()
    if peak > 0: waveform /= peak
    # RMS-Normalize
    rms = np.sqrt(np.mean(waveform ** 2))
    if rms > 0:
        target_linear = 10 ** (cfg.TARGET_DB / 20.0)
        waveform *= (target_linear / rms)
        waveform = np.clip(waveform, -1.0, 1.0)
    return waveform

def chunk_waveform(waveform: np.ndarray) -> List[np.ndarray]:
    step = int(cfg.CHUNK_DURATION * cfg.SAMPLE_RATE)
    min_samples = int(cfg.MIN_DURATION_SEC * cfg.SAMPLE_RATE)
    chunks = []
    for start in range(0, len(waveform), step):
        segment = waveform[start : start + step]
        if len(segment) >= min_samples:
            if len(segment) < step:
                segment = np.pad(segment, (0, step - len(segment)))
            chunks.append(segment)
    return chunks

def process_directory(src_dir: Union[str, Path], dst_dir: Path, label: str):
    src = Path(src_dir)
    if not src.exists():
        log.warning(f"Path does not exist: {src_dir}")
        return 0
    
    files = sorted([f for f in src.rglob("*") if f.suffix.lower() in [".wav", ".mp3", ".flac"]])
    if not files: return 0
    
    dst_dir.mkdir(parents=True, exist_ok=True)
    total_chunks = 0
    
    for f in tqdm(files, desc=f"Processing {label}"):
        try:
            wav = load_audio(f)
            wav = transform_audio(wav)
            chunks = chunk_waveform(wav)
            for i, chunk in enumerate(chunks):
                out = dst_dir / f"{label}_{f.stem}_c{i:03d}.wav"
                sf.write(str(out), chunk, cfg.SAMPLE_RATE, subtype="PCM_16")
                total_chunks += 1
        except Exception as e:
            continue
    return total_chunks

print("✅ Preprocessing engine loaded.")

In [ ]:
# @title 📥 Step 4: Data Collection
def download_hf(dataset_id, split="train", label="real", max_samples=500):
    from datasets import load_dataset, Audio
    import io
    out_dir = cfg.RAW_DIR / label
    out_dir.mkdir(parents=True, exist_ok=True)
    
    ds = load_dataset(dataset_id, split=split, streaming=True, trust_remote_code=True)
    if "audio" in ds.column_names: ds = ds.cast_column("audio", Audio(decode=False))
    
    saved = 0
    for i, sample in enumerate(tqdm(ds, desc=f"Downloading {label}", total=max_samples)):
        if saved >= max_samples: break
        audio = sample.get("audio") or sample.get("speech")
        try:
            if isinstance(audio, dict) and "bytes" in audio and audio["bytes"]:
                arr, sr = sf.read(io.BytesIO(audio["bytes"]))
            else:
                arr, sr = librosa.load(audio["path"], sr=cfg.SAMPLE_RATE)
            
            if sr != cfg.SAMPLE_RATE: arr = librosa.resample(arr, orig_sr=sr, target_sr=cfg.SAMPLE_RATE)
            sf.write(str(out_dir/f"hf_{i:06d}.wav"), arr, cfg.SAMPLE_RATE)
            saved += 1
        except: continue
    return saved

# 1. Download Public Data
print("--- Downloading Public Datasets ---")
download_hf("google/fleurs", "ta_in.test", "real", 100) # Tamil Real
download_hf("dkounadis/artificial-speech", "train", "fake", 200) # Fake

# 2. Run Preprocessing on Raw Public Data
print("--- Preprocessing Public Data ---")
process_directory(cfg.RAW_DIR/"real", cfg.PROCESSED_DIR/"real", "real")
process_directory(cfg.RAW_DIR/"fake", cfg.PROCESSED_DIR/"fake", "fake")

# 3. Run Preprocessing on CUSTOM DATASETS
print("--- Preprocessing Custom Datasets ---")
if cfg.CUSTOM_PATH_1:
    n = process_directory(cfg.CUSTOM_PATH_1, cfg.PROCESSED_DIR / cfg.CUSTOM_LABEL_1, f"custom1_{cfg.CUSTOM_LABEL_1}")
    print(f"✅ Custom Dataset 1: {n} chunks processed.")

if cfg.CUSTOM_PATH_2:
    n = process_directory(cfg.CUSTOM_PATH_2, cfg.PROCESSED_DIR / cfg.CUSTOM_LABEL_2, f"custom2_{cfg.CUSTOM_LABEL_2}")
    print(f"✅ Custom Dataset 2: {n} chunks processed.")

# Build Final Metadata
rows = []
for label in ["real", "fake"]:
    for f in (cfg.PROCESSED_DIR / label).glob("*.wav"):
        rows.append({"path": str(f), "label": label, "label_id": cfg.INV_LABEL_MAP[label]})
df_meta = pd.DataFrame(rows)
df_meta.to_csv(cfg.PROCESSED_DIR / "metadata.csv", index=False)
print(f"\n🚀 Total training chunks: {len(df_meta)}")

In [ ]:
# @title 🧠 Step 5: Model Architecture
from transformers import Wav2Vec2Model, Wav2Vec2PreTrainedModel, Wav2Vec2Config, Wav2Vec2FeatureExtractor
from transformers.modeling_outputs import SequenceClassifierOutput

class ClassificationHead(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(cfg.CLASSIFIER_DROPOUT),
            nn.Linear(256, cfg.NUM_LABELS),
        )
    def forward(self, x): return self.net(x)

class DeepfakeAudioDetector(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.wav2vec2 = Wav2Vec2Model(config)
        self.classifier = ClassificationHead(config.hidden_size)
        self.init_weights()
    
    def forward(self, input_values, attention_mask=None, labels=None):
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state.mean(dim=1)
        logits = self.classifier(pooled)
        loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)

def build_model():
    w2v_cfg = Wav2Vec2Config.from_pretrained(cfg.MODEL_NAME, num_labels=cfg.NUM_LABELS)
    w2v_cfg.hidden_dropout = cfg.HIDDEN_DROPOUT
    model = DeepfakeAudioDetector.from_pretrained(cfg.MODEL_NAME, config=w2v_cfg, ignore_mismatched_sizes=True)
    if cfg.FREEZE_FEATURE_EXTRACTOR: model.wav2vec2.feature_extractor._freeze_parameters()
    return model

print("✅ Model architecture defined.")

In [ ]:
# @title 📊 Step 6: Dataset & Metrics
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

class AudioDataset(Dataset):
    def __init__(self, records, feature_extractor):
        self.records = records
        self.fe = feature_extractor
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        rec = self.records[idx]
        wav, _ = librosa.load(rec["path"], sr=cfg.SAMPLE_RATE)
        enc = self.fe(wav, sampling_rate=cfg.SAMPLE_RATE, return_tensors="pt", padding="max_length", max_length=int(cfg.CHUNK_DURATION*cfg.SAMPLE_RATE), truncation=True)
        return {"input_values": enc.input_values.squeeze(0), "labels": torch.tensor(rec["label_id"])}

fe = Wav2Vec2FeatureExtractor.from_pretrained(cfg.MODEL_NAME)
train_recs, val_recs = train_test_split(df_meta.to_dict("records"), test_size=cfg.VAL_SPLIT, random_state=cfg.SEED, stratify=df_meta["label_id"])

train_loader = DataLoader(AudioDataset(train_recs, fe), batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(AudioDataset(val_recs, fe), batch_size=cfg.EVAL_BATCH_SIZE)

print(f"✅ DataLoaders created. Train: {len(train_recs)}, Val: {len(val_recs)}")

In [ ]:
# @title 🚀 Step 7: Training Loop
from transformers import get_cosine_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)
total_steps = len(train_loader) * cfg.NUM_EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps*cfg.WARMUP_RATIO), total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=cfg.FP16)

best_f1 = 0
for epoch in range(cfg.NUM_EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()
        inputs, labels = batch["input_values"].to(device), batch["labels"].to(device)
        with torch.cuda.amp.autocast(enabled=cfg.FP16):
            outputs = model(inputs, labels=labels)
            loss = outputs.loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
    
    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            inputs, labels = batch["input_values"].to(device), batch["labels"].to(device)
            logits = model(inputs).logits
            all_preds.extend(torch.argmax(logits, -1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    
    f1 = f1_score(all_labels, all_preds)
    acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Val F1: {f1:.4f} | Acc: {acc:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained(cfg.CHECKPOINTS / "best_model")
        fe.save_pretrained(cfg.CHECKPOINTS / "best_model")
        print("⭐ New best model saved!")

In [ ]:
# @title 📊 Step 8: Comprehensive Evaluation
from sklearn.metrics import confusion_matrix, classification_report

def run_final_eval():
    best_model = DeepfakeAudioDetector.from_pretrained(cfg.CHECKPOINTS / "best_model").to(device)
    best_model.eval()
    preds, targets = [], []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Final Eval"):
            inputs, labels = batch["input_values"].to(device), batch["labels"].to(device)
            logits = best_model(inputs).logits
            preds.extend(torch.argmax(logits, -1).cpu().tolist())
            targets.extend(labels.cpu().tolist())
    
    cm = confusion_matrix(targets, preds)
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=cfg.LABEL_MAP.values(), yticklabels=cfg.LABEL_MAP.values())
    plt.title("Confusion Matrix")
    plt.show()
    
    print("\nClassification Report:")
    print(classification_report(targets, preds, target_names=cfg.LABEL_MAP.values()))

run_final_eval()

In [ ]:
# @title 🧪 Step 9: Inference Wrapper
class DeepfakeDetector:
    def __init__(self, model_dir):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = DeepfakeAudioDetector.from_pretrained(model_dir).to(self.device).eval()
        self.fe = Wav2Vec2FeatureExtractor.from_pretrained(model_dir)
    
    def predict(self, path):
        wav = load_audio(path)
        wav = transform_audio(wav)
        chunks = chunk_waveform(wav)
        
        results = []
        for chunk in chunks:
            enc = self.fe(chunk, sampling_rate=cfg.SAMPLE_RATE, return_tensors="pt").input_values.to(self.device)
            with torch.no_grad():
                logits = self.model(enc).logits
                probs = torch.softmax(logits, -1)[0]
                results.append(probs.cpu().numpy())
        
        avg_probs = np.mean(results, axis=0)
        pred = np.argmax(avg_probs)
        return cfg.LABEL_MAP[pred], avg_probs[pred], avg_probs

# Interactive Test
detector = DeepfakeDetector(cfg.CHECKPOINTS / "best_model")
sample = random.choice(val_recs)
label, conf, p = detector.predict(sample["path"])

res_table = Table(title="Inference Result")
res_table.add_column("Audio File", style="cyan")
res_table.add_column("Prediction", style="bold green" if label=="real" else "bold red")
res_table.add_column("Confidence", justify="right")
res_table.add_row(Path(sample['path']).name, label.upper(), f"{conf:.2%}")
console.print(res_table)